# BharatNanoGPT — Full 14M Run on Colab T4

**Kuch type nahi karna. Bas ye notebook upload karo, Runtime > T4 GPU select karo, phir Runtime > Run all dabao.** Notebook khud sab karega: repo clone → fixed files download → GPU check → install → prebuilt corpus (120,280 tokens) → train (~14M params, 6000 iters, 1–2 ghante) → evaluate → samples → checkpoint download.

Wikipedia Colab pe HTTP 429 deta hai, isliye notebook Wikipedia fetch nahi karta — prebuilt corpus download karta hai. Training khatam hone tak tab khula rakho.


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — Runtime > Change runtime type > T4 GPU karo, phir dobara chalao')


In [ ]:
!git clone https://github.com/SidakSethi-Singh/BharatNanoGPT.git
%cd BharatNanoGPT
!ls

In [ ]:
# Fixed files download karo (GitHub wale purane hain — inme 429 retry,
# --from-text aur corpus-too-small ka clear error hai)
!wget -q 'https://muse.ai/files/1514689061718027/1861672818299367/cvgb3xjt997frpy1ypt79hkw/prepare.py' -O data/prepare.py
!wget -q 'https://muse.ai/files/1514689061718027/3243471925847626/n2g3gowwl0n3fwkyyq8nbtan/train.py' -O train.py
!ls -lh data/prepare.py train.py


In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## Corpus banao (bada wala)
Wikipedia Colab se aksar HTTP 429 deta hai, isliye notebook prebuilt corpus (120,280 tokens) download karta hai — dobara Wikipedia fetch karne ki zaroorat nahi. Agar `Tokens: total` 100,000 se kam dikhe, to train mat karo, mujhe batao.


In [ ]:
# Wikipedia Colab pe HTTP 429 (Too Many Requests) deta hai, isliye
# prebuilt corpus (120,280 tokens, 1,139 lines) download karo — ye wahi
# corpus hai jisse tiny model train hua tha.
!wget -q 'https://muse.ai/files/1514689061718027/1592707635952443/y7cgwhrdahwogjmra3cwaint/corpus.txt' -O /tmp/corpus_full.txt && ls -lh /tmp/corpus_full.txt
!python -m data.prepare --config config/train_bharat.py --from-text /tmp/corpus_full.txt
!ls -lh data/


## Train karo — full 14M model
Ye cell sabse lamba chalega (1–2 ghante, T4 pe). Loss har 50 iter pe dikhega. Val loss 500 iter pe check hota hai. Agar val loss badhne lage to training rok do — checkpoint `out/bharat/ckpt.pt` me best wala hi use karna.

In [ ]:
!python train.py --config config/train_bharat.py

## Evaluate karo — perplexity + real samples

In [ ]:
!python evaluate.py --checkpoint out/bharat/ckpt.pt --num-samples 6
!python sample.py --checkpoint out/bharat/ckpt.pt --prompt "राम सुबह उठकर" --temperature 0.8 --top-k 40 --max-new-tokens 60
!python sample.py --checkpoint out/bharat/ckpt.pt --prompt "bhai kal exam hai aur mujhe" --temperature 0.8 --top-k 40 --max-new-tokens 60
!python sample.py --checkpoint out/bharat/ckpt.pt --prompt "The capital of India is" --temperature 0.8 --top-k 40 --max-new-tokens 60

## Checkpoint download karo
Ye file demo ke liye chahiye: `out/bharat/ckpt.pt` (~55 MB). Ise apne laptop pe save karo, phir INFRASHIFT demo me `out/bharat/ckpt.pt` ke roop me use karna. `results/eval_samples.jsonl` bhi download kar lo — README me dalne ke liye.

In [ ]:
!ls -lh out/bharat/ assets/ results/ 2>/dev/null
from google.colab import files
files.download('out/bharat/ckpt.pt')
files.download('results/eval_samples.jsonl')

## Agar kuch gadbad ho
- **GPU nahi mila:** Runtime > Change runtime type > T4 GPU, phir Runtime > Restart and run all.
- **Session disconnect:** Dobara clone mat karo — Drive mount karo ya `!git pull` karke `--init-from out/bharat/ckpt.pt` se resume karo (train.py me resume support hai).
- **Loss nahi gir raha:** Pehle 100 iter me 7.6 se ~5 tak girna chahiye; nahi gira to corpus check karo (`data/train.bin` size).
- **Colab free limit khatam:** Subah dobara try karo, ya tiny config se demo chalao — demo checkpoint pehle se repo me hai.